# **任务13 图神经网络 - 节点分类 | GNN - Node Classification**

对图中每个节点进行分类：对于关系网中每个个体进行分类

## 1. 模型定义

还记得本次任务最开始展示的图卷积层是如何工作的，它的输出依然是按节点的特征向量，形状为 `[batch_size * node_dim, node_features]` , 如果我们将输出的 `node_features` 映射为类别个数，就可以给每个节点的每个类别一个概率得分了。

由于图神经网络结构的特殊性，不能在输入模型之前移动到GPU设备上，如果在生成数据时转移，则可能导致显存大量占用，所以这里需要在模型中添加将张量同步转移到GPU的代码。


In [9]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

class ModelNode(nn.Module):
    def __init__(self, num_node_features, num_classes):
        super(ModelNode, self).__init__()
        self.conv1 = GCNConv(num_node_features, 16)
        self.conv2 = GCNConv(16, 32)
        self.conv3 = GCNConv(32, num_classes)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        device = next(self.parameters()).device
        x, edge_index = x.to(device), edge_index.to(device)
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        x = self.conv2(x, edge_index)
        x = self.relu(x)
        x = self.conv3(x, edge_index)
        return x

## 2. 数据生成

边索引为用户之间的好友关系。模拟判断某个用户经常访问某个分区，来判断给其推荐哪些内容，0:体育 1:首饰 2:家居 3:游戏 4:科技 5:书籍

一共 `node_num=1200` 个节点，也就是一个群体中的用户数量。

In [10]:
import random
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def get_data(data_size):
    graph_datas = []
    for _ in range(data_size):
        x = []
        y = []
        # 数据生成
        node_num = 1200
        for n in range(node_num):
            max_num = random.randint(0, 5)

            num_1 = random.randint(0, 100)
            num_2 = random.randint(0, 100 - num_1)
            num_3 = random.randint(0, 100 - num_1 - num_2)
            num_4 = random.randint(0, 100 - num_1 - num_2 - num_3)
            num_5 = random.randint(0, 100 - num_1 - num_2 - num_3 - num_4)
            num_6 = 100 - num_1 - num_2 - num_3 - num_4 - num_5
            if max_num == 0:
                x_list = [num_1, num_2, num_3, num_4, num_5, num_6]
            elif max_num == 1:
                x_list = [num_2, num_1, num_3, num_4, num_5, num_6]
            elif max_num == 2:
                x_list = [num_3, num_2, num_1, num_4, num_5, num_6]
            elif max_num == 3:
                x_list = [num_4, num_2, num_3, num_1, num_5, num_6]
            elif max_num == 4:
                x_list = [num_5, num_2, num_3, num_4, num_1, num_6]
            else:
                x_list = [num_6, num_2, num_3, num_4, num_5, num_1]

            x.append(x_list)
            y_num = x_list.index(max(x_list))
            x_list[y_num] += 50
            y.append(y_num)

        x = torch.tensor(x, dtype=torch.float)
        y = torch.tensor(y, dtype=torch.long)
        edge = [[], []]
        for n in range(120):
            a, b = random.choices(list(range(0, node_num-1)), k=2)
            edge[0].append(a)
            edge[1].append(b)
        edge_index = torch.tensor(edge, dtype=torch.long)

        data2 = Data(x=x, edge_index=edge_index, y=y)
        graph_datas.append(data2)
    return graph_datas

# 训练数据
batch_size = 32
dataset = get_data(640)
cut_num = int(len(dataset)*0.7)
# 分离训练集和验证集
train_dataset = dataset[:cut_num]
val_dataset = dataset[cut_num:]
train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

for batch in train_loader:
    print("批次数据信息：", batch)
    break

批次数据信息： DataBatch(x=[38400, 6], edge_index=[2, 3840], y=[38400], batch=[38400], ptr=[33])


## 3. 模型训练

### 3.1 实例化模型、损失函数、优化器

任务是二元分类问题，使用交叉熵损失 `nn.CrossEntropyLoss()`。

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_node_features = dataset[0].num_node_features
num_classes = 6

model = ModelNode(num_node_features, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 3.2 **迭代训练**

In [12]:
epochs = 60
# 训练模型
model.train()
for epoch in range(epochs):
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    # 验证
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch)
            loss = criterion(out.view(-1, num_classes), batch.y.view(-1))
            val_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += int((pred == batch.y).sum())
            total += batch.y.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total

    if (epoch + 1) % 12 == 0:
        print(f'Epoch[{epoch + 1}] Train Loss: {train_loss}')
        print(f'\tVal Loss: {val_loss} \tVal Acc: {val_acc}')

Epoch[12] Train Loss: 0.6337579488754272
	Val Loss: 0.6195949415365855 	Val Acc: 0.9118012152777778
Epoch[24] Train Loss: 0.47259776294231415
	Val Loss: 0.4704020420710246 	Val Acc: 0.9233072916666667
Epoch[36] Train Loss: 0.43785296167646137
	Val Loss: 0.43721526364485425 	Val Acc: 0.9238411458333333
Epoch[48] Train Loss: 0.41854532275881084
	Val Loss: 0.41811686257521313 	Val Acc: 0.9239713541666666
Epoch[60] Train Loss: 0.40625585189887453
	Val Loss: 0.40616682668526966 	Val Acc: 0.9239105902777778


## 3. 总结

这是第二个图结构的任务，表明图结构的应用可以更具想象力。